# Monetization Estimation

Revenue projections built on top of the retention/DAU simulation.

In [1]:
from IPython.display import HTML
display(HTML('''
<style id="aco-toggle-style"></style>
<button id="aco-toggle-btn"
  onclick="
    var style = document.getElementById('aco-toggle-style');
    if (style.innerHTML === '') {
      style.innerHTML = '.jp-Cell-inputWrapper { display: none !important; }';
      this.innerHTML = 'Show Code';
    } else {
      style.innerHTML = '';
      this.innerHTML = 'Hide Code';
    }
  "
  style="padding: 6px 16px; font-size: 13px; cursor: pointer;
         border: 1px solid #ccc; border-radius: 4px; background: #f5f5f5;">
  Hide Code
</button>
'''))

In [2]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from pathlib import Path

if Path.cwd().name == 'notebooks':
    os.chdir(Path.cwd().parent)

from aco_model.models import MonetizationParams, RetentionCurve
from aco_model.retention import load_installs, retention_vector, simulate
from aco_model.monetization import estimate_revenue
from aco_model.config import load_config
from aco_model.state import load_state, save_state, DEFAULT_STATE_PATH

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

_HEADLESS = os.environ.get('ACO_HEADLESS') == '1'

cfg = load_config()
state = load_state()

if state is not None:
    installs = load_installs(cfg.installs_path)
    curve = RetentionCurve(anchors=state.retention_anchors)
    sim = simulate(installs, curve, state.sim_days)
    monetization = state.monetization
    print(f'=== Loaded from shared state ({DEFAULT_STATE_PATH}) ===')
    print(f'  Updated: {state.updated_at}')
else:
    installs = load_installs(cfg.installs_path)
    sim = simulate(installs, cfg.retention, cfg.sim_days)
    monetization = cfg.monetization
    print(f'=== No state file — using config.yaml defaults ===')

print(f'  DAU range: {sim.dau.min():,} – {sim.dau.max():,}')
print(f'  Monetization: {monetization.pct_payers:.1%} payers, ${monetization.arppu:.2f} ARPPU')

=== Loaded from shared state (output/state.json) ===
  Updated: 2026-03-30T18:02:28
  DAU range: 15,182 – 62,083
  Monetization: 3.0% payers, $1.50 ARPPU


## 1. Revenue Estimation

In [3]:
payers_slider = widgets.FloatSlider(value=monetization.pct_payers * 100, min=0.5, max=15.0, step=0.5,
                                     description='Payers %:')
arppu_slider = widgets.FloatSlider(value=monetization.arppu, min=0.25, max=50.0, step=0.25,
                                    description='ARPPU $:')

_current_anchors = state.retention_anchors if state else cfg.retention.anchors
_mon_slider_args = {'payers_pct': payers_slider, 'arppu': arppu_slider}

def _make_params(payers_pct, arppu):
    return MonetizationParams(pct_payers=payers_pct / 100.0, arppu=arppu)

def plot_revenue(payers_pct, arppu):
    params = _make_params(payers_pct, arppu)
    rev = estimate_revenue(sim, params)
    df = rev.to_dataframe()
    save_state(sim, _current_anchors, monetization=params)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    axes[0].fill_between(df['day'], df['daily_revenue_usd'], alpha=0.3, color='#4CAF50')
    axes[0].plot(df['day'], df['daily_revenue_usd'], linewidth=2, color='#4CAF50')
    axes[0].set_xlabel('Day'); axes[0].set_ylabel('Daily Revenue (USD)'); axes[0].set_title('Daily Revenue')

    axes[1].plot(df['day'], df['cumulative_revenue_usd'], linewidth=2, color='#FF5722')
    axes[1].set_xlabel('Day'); axes[1].set_ylabel('Cumulative Revenue (USD)'); axes[1].set_title('Cumulative Revenue')

    axes[2].plot(df['day'], df['arpdau_usd'], linewidth=2, color='#9C27B0')
    axes[2].set_xlabel('Day'); axes[2].set_ylabel('ARPDAU (USD)'); axes[2].set_title('ARPDAU')
    plt.tight_layout(); plt.show()

    print(f'ARPDAU: ${params.pct_payers * params.arppu:,.4f}')
    print(f'Avg Lifetime Rev per Payer: ${rev.avg_lifetime_revenue_per_payer:,.2f}')
    print(f'Total Payers: {rev.total_payers:,} | Total Revenue: ${rev.total_revenue:,.2f}')
    print(f'Avg Daily Revenue: ${rev.total_revenue / sim.sim_days:,.2f} | Peak: ${rev.daily_revenue.max():,.2f}')
    print(f'Avg Revenue per Cohort: ${rev.avg_revenue_per_cohort:,.2f}')

if _HEADLESS:
    plot_revenue(payers_slider.value, arppu_slider.value)
else:
    out_rev = widgets.interactive_output(plot_revenue, _mon_slider_args)
    display(widgets.VBox([widgets.HBox([payers_slider, arppu_slider]), out_rev]))

## 2. Revenue Sensitivity

In [4]:
def plot_sensitivity(payers_pct, arppu):
    base_pct = payers_pct / 100.0
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    pct_values = sorted(set([1, 2, 3, 5, 8, payers_pct]))
    for pct in pct_values:
        r = estimate_revenue(sim, MonetizationParams(pct_payers=pct/100, arppu=arppu))
        style = {'linewidth': 3, 'linestyle': '--'} if pct == payers_pct else {'linewidth': 2}
        ax1.plot(np.arange(1, sim.sim_days + 1), r.cumulative_revenue, label=f'{pct}% payers', **style)
    ax1.set_xlabel('Day'); ax1.set_ylabel('Cumulative Revenue (USD)')
    ax1.set_title(f'Sensitivity: % Payers (ARPPU=${arppu:.2f})'); ax1.legend()

    arppu_values = sorted(set([0.50, 1.00, 1.50, 2.50, 5.00, arppu]))
    for a in arppu_values:
        r = estimate_revenue(sim, MonetizationParams(pct_payers=base_pct, arppu=a))
        style = {'linewidth': 3, 'linestyle': '--'} if a == arppu else {'linewidth': 2}
        ax2.plot(np.arange(1, sim.sim_days + 1), r.cumulative_revenue, label=f'${a:.2f}', **style)
    ax2.set_xlabel('Day'); ax2.set_ylabel('Cumulative Revenue (USD)')
    ax2.set_title(f'Sensitivity: ARPPU ({payers_pct:.1f}% Payers)'); ax2.legend()
    plt.tight_layout(); plt.show()

if _HEADLESS:
    plot_sensitivity(payers_slider.value, arppu_slider.value)
else:
    out_sens = widgets.interactive_output(plot_sensitivity, _mon_slider_args)
    display(out_sens)

Output()

## 3. Revenue per Install Cohort

In [5]:
def plot_cohort_revenue(payers_pct, arppu):
    params = _make_params(payers_pct, arppu)
    rev = estimate_revenue(sim, params)
    cdf = rev.cohort_revenue_dataframe()

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    ax1.bar(cdf['cohort_day'], cdf['lifetime_revenue_usd'], alpha=0.7, color='#4CAF50')
    ax1.set_xlabel('Cohort (Install Day)'); ax1.set_ylabel('Lifetime Revenue (USD)')
    ax1.set_title('Total Revenue by Install Cohort')

    ax2.bar(cdf['cohort_day'], cdf['revenue_per_install_usd'], alpha=0.7, color='#FF9800')
    ax2.set_xlabel('Cohort (Install Day)'); ax2.set_ylabel('Revenue per Install (USD)')
    ax2.set_title('Revenue per Install by Cohort')
    plt.tight_layout(); plt.show()

    print(f'Avg Lifetime Revenue per Cohort: ${rev.avg_revenue_per_cohort:,.2f}')
    print(f'Avg Revenue per Install: ${cdf["revenue_per_install_usd"].mean():,.4f}')
    print(f'Best Cohort: Day {cdf.loc[cdf["lifetime_revenue_usd"].idxmax(), "cohort_day"]} (${cdf["lifetime_revenue_usd"].max():,.2f})')

if _HEADLESS:
    plot_cohort_revenue(payers_slider.value, arppu_slider.value)
else:
    out_cohort = widgets.interactive_output(plot_cohort_revenue, _mon_slider_args)
    display(out_cohort)

Output()

## 4. Combined DAU + Revenue View

In [6]:
def plot_combined(payers_pct, arppu):
    params = _make_params(payers_pct, arppu)
    rev = estimate_revenue(sim, params)
    days = np.arange(1, sim.sim_days + 1)

    fig, ax1 = plt.subplots(figsize=(12, 5))
    ax1.fill_between(days, sim.dau, alpha=0.2, color='#2196F3')
    ax1.plot(days, sim.dau, linewidth=2, color='#2196F3', label='DAU')
    ax1.set_xlabel('Day'); ax1.set_ylabel('DAU', color='#2196F3')
    ax1.tick_params(axis='y', labelcolor='#2196F3')

    ax2 = ax1.twinx()
    ax2.plot(days, rev.daily_revenue, linewidth=2, color='#4CAF50', label='Daily Revenue')
    ax2.set_ylabel('Daily Revenue (USD)', color='#4CAF50')
    ax2.tick_params(axis='y', labelcolor='#4CAF50')

    ax1.set_title(f'DAU + Revenue ({payers_pct:.1f}% payers, ${arppu:.2f} ARPPU)')
    fig.legend(loc='upper left', bbox_to_anchor=(0.12, 0.88))
    plt.tight_layout(); plt.show()

if _HEADLESS:
    plot_combined(payers_slider.value, arppu_slider.value)
else:
    out_combined = widgets.interactive_output(plot_combined, _mon_slider_args)
    display(out_combined)

Output()